# Content-Based Filtering Pipeline

To answer the question: Given the games a user has positively reviewed in the past and the way they express their preferences in their own words, what games are most similar to both their play history and their stated tastes?

It is a hybrid model - 60% game history and 40% taste profile side that answers "find me games that match how I think and talk about games"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install sentence-transformers

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

## Importing libraries

In [ ]:
#Data
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import shutil
from tqdm import tqdm
import numpy as np

game_features = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/game_features_static.parquet')
game_descriptions = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/game_description_embeddings.parquet')
train_reviews = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/train_with_features.parquet')
test_reviews = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/test_with_features.parquet')

print(f"Train: {len(train_reviews)} reviews")
print(f"Test:  {len(test_reviews)} reviews")
print(f"Train %: {len(train_reviews)/(len(train_reviews) + len(test_reviews))*100:.1f}%")

Train: 173832 reviews
Test:  51063 reviews
Train %: 77.3%


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

KeyboardInterrupt: 

In [ ]:
game_features.head()
game_descriptions.head()

,desc_emb_0,desc_emb_1,desc_emb_2,desc_emb_3,desc_emb_4,desc_emb_5,desc_emb_6,desc_emb_7,desc_emb_8,desc_emb_9,...,desc_emb_118,desc_emb_119,desc_emb_120,desc_emb_121,desc_emb_122,desc_emb_123,desc_emb_124,desc_emb_125,desc_emb_126,desc_emb_127
appid,,,,,,,,,,,,,,,,,,,,,
400,0.127334,0.234454,-0.045320,0.093814,-0.060112,0.101858,-0.007598,0.076641,0.199637,0.011200,...,-0.057913,-0.013363,0.024660,-0.008150,0.044710,0.019605,0.006236,0.006766,0.029913,-0.006292
1300,-0.086652,-0.266915,0.037698,0.010966,-0.101552,0.004867,0.140006,0.115462,-0.047205,0.029091,...,-0.023922,-0.066933,0.078879,0.011939,-0.004152,-0.063366,-0.086202,-0.039160,-0.006131,-0.008258
1313,-0.104068,-0.056749,0.135682,-0.094452,-0.048875,0.034507,-0.059555,0.056446,-0.020930,-0.129082,...,0.022062,0.018692,0.004197,0.002764,0.008816,-0.084858,-0.020806,-0.010951,0.092830,-0.002066
1520,0.202559,0.001342,-0.223662,0.068996,-0.229597,0.005506,0.216479,0.033829,-0.045168,0.028776,...,-0.003956,-0.023499,0.117325,0.035725,0.018913,-0.002188,0.109917,-0.000737,-0.020540,0.060800
1640,-0.036931,-0.078148,-0.158223,-0.169378,-0.118772,0.170498,0.060643,0.142795,0.003209,-0.102198,...,-0.006402,-0.100778,-0.068321,-0.082844,-0.017296,0.000266,0.010155,-0.016985,0.115138,-0.097554


## Game feature construction

Merge static game features (genre, category, ...) with BERT description embedding into a single per-game vector. This represents what each game is.

In [ ]:
#run for tests
game_vectors = pd.merge(game_features, game_descriptions, on='appid')
print(game_vectors.shape)

(32883, 222)


## Community Sentiment

For each game, compute two things from training reviews: a sentiment score (ratio of positive reviews) and an average BERT embedding of all review text written about that game. This captures what players say about the game beyond the official description.

In [ ]:
from textblob import TextBlob
import os

# 1. Sentiment score — voted_up ratio per game
community_sentiment = (
    train_reviews
    .groupby('appid')
    .agg(
        sentiment_score=('voted_up', 'mean'),
        review_count=('voted_up', 'count')
    )
    .reset_index()
)

# 2. Average BERT embedding per game — encode once, aggregate by group
valid_train = (
    train_reviews[train_reviews['review_text'].notna() &
                  (train_reviews['review_text'].str.strip() != '')]
    .copy()
    .reset_index(drop=True)
)

all_review_embeddings = model.encode(
    valid_train['review_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

valid_train['emb_idx'] = range(len(valid_train))


game_review_embs = {}
for appid, group in tqdm(valid_train.groupby('appid')): # ← renamed
    idx = group['emb_idx'].values
    game_review_embs[appid] = all_review_embeddings[idx].mean(axis=0)

# 3. Assemble into dataframe and merge
emb_records = [
    {'appid': appid, **{f'review_emb_{i}': val for i, val in enumerate(emb)}}
    for appid, emb in game_review_embs.items()
]
df_review_embs = pd.DataFrame(emb_records)

df_community = community_sentiment.merge(df_review_embs, on='appid', how='left')
print(f"Community sentiment shape: {df_community.shape}")
print(f"Games with sentiment: {len(df_community)}")

Batches:   0%|          | 0/679 [00:00<?, ?it/s]

100%|██████████| 31544/31544 [00:04<00:00, 7727.72it/s]


Community sentiment shape: (31551, 387)
Games with sentiment: 31551


In [ ]:
# df_community.to_parquet('/content/drive/MyDrive/Phase 2/community_sentiment.parquet', index=False)
# print("Saved ✅")

Saved ✅


## User Taste Profiles

For each user, build a recency-weighted average of the BERT embeddings of their own reviews. More recent reviews are weighted higher. This captures how the user talks about games.

In [ ]:
# Build per-user average review embedding
user_taste_raw = {}
for user_id, group in tqdm(valid_train.groupby('author_steamid')):
    idx = group['emb_idx'].values
    timestamps = group['timestamp_created'].values.astype(float)

    vecs = all_review_embeddings[idx]

    # Recency weights
    t_max = timestamps.max()
    deltas = t_max - timestamps
    weights = np.exp(-0.001 * deltas)
    weights = weights / weights.sum()

    user_taste_raw[user_id] = np.average(vecs, axis=0, weights=weights)

print(f"User taste profiles built: {len(user_taste_raw)}")

100%|██████████| 36144/36144 [00:09<00:00, 3701.86it/s]

User taste profiles built: 36144


## Feature Matrix

This matrix combines all game features, scales structured features to [0,1], reduces embeddings with PCA to 32 dims, upweights structured features 3x to prevent embeddings from dominating, then normalises row-wise for cosine similarity

In [ ]:
# PCA to reduce to 32 dims — same as desc_pca
user_ids_list = list(user_taste_raw.keys())
user_matrix = np.stack([user_taste_raw[u] for u in user_ids_list])

pca_user = PCA(n_components=32)
user_matrix_32 = pca_user.fit_transform(user_matrix)

user_taste_profiles = {
    user_id: user_matrix_32[i]
    for i, user_id in enumerate(user_ids_list)
}
print("User profiles reduced to 32 dims")

User profiles reduced to 32 dims


In [ ]:
user_taste_df = pd.DataFrame([
    {'author_steamid': user_id, **{f'taste_emb_{i}': val for i, val in enumerate(vec)}}
    for user_id, vec in user_taste_profiles.items()
])
user_taste_df.to_parquet('/content/drive/MyDrive/Phase 2/user_taste_profiles.parquet', index=False)
print("Saved ✅")

Saved ✅


## Recommendation

For each user, blend two signals: 60% from the average feature vector of games they positively reviewed in train (what they played) and 40% from their personal taste profile (how they talked about games). This blended query vector is compared against all games via cosine similarity to return top-10 game recommendations, excluding already seen games

In [ ]:
# df_community = pd.read_parquet('/content/drive/MyDrive/Phase 2/data/community_sentiments.parquet')
# print(df_community.shape)
# print(df_community.columns.tolist()[:10])

In [ ]:
#run for tests
df_community = pd.read_parquet('/content/drive/MyDrive/phase2_finalised/Phase 2.2. Enhanced Model/community_sentiment.parquet')
user_taste_df = pd.read_parquet('/content/drive/MyDrive/phase2_finalised/Phase 2.2. Enhanced Model/user_taste_profiles.parquet')

user_taste_profiles = {
    row['author_steamid']: np.array([row[f'taste_emb_{i}'] for i in range(32)])
    for _, row in user_taste_df.iterrows()
}

In [ ]:
#run for tests
game_vectors = game_vectors.merge(df_community[['appid', 'sentiment_score', 'review_count']],
                                   on='appid', how='left')
game_vectors['sentiment_score'] = game_vectors['sentiment_score'].fillna(0.5)
game_vectors['review_count'] = game_vectors['review_count'].fillna(0)

In [ ]:
#run for tests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize, MinMaxScaler
import os
import tqdm

train_liked = (
    train_reviews[train_reviews['voted_up'] == True]
    .groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
train_liked.columns = ['author_steamid', 'seed_appids']
print(f"Users with liked games in train: {len(train_liked)}")

OUTPUT_DIR = '/content/drive/MyDrive/Phase 2'

if 'appid' not in game_vectors.columns:
    game_vectors = game_vectors.reset_index()

feature_cols = game_vectors.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in feature_cols if c not in ['appid', 'level_0', 'index']]

structured_cols = [c for c in feature_cols if not c.startswith(('desc_emb_', 'review_emb_', 'user_emb_'))]
embedding_cols = [c for c in feature_cols if c.startswith(('desc_emb_', 'review_emb_'))]

scaler = MinMaxScaler()
game_vectors[structured_cols] = scaler.fit_transform(game_vectors[structured_cols])

desc_cols = [c for c in feature_cols if c.startswith('desc_emb_')]
review_cols = [c for c in feature_cols if c.startswith('review_emb_')]

if desc_cols:
    pca_desc = PCA(n_components=32)
    desc_pca = pca_desc.fit_transform(game_vectors[desc_cols])
    for i in range(32):
        game_vectors[f'desc_pca_{i}'] = desc_pca[:, i]

if review_cols:
    pca_review = PCA(n_components=32)
    review_pca = pca_review.fit_transform(game_vectors[review_cols])
    for i in range(32):
        game_vectors[f'review_pca_{i}'] = review_pca[:, i]

pca_cols = []
if desc_cols:
    pca_cols += [f'desc_pca_{i}' for i in range(32)]
if review_cols:
    pca_cols += [f'review_pca_{i}' for i in range(32)]
feature_cols = structured_cols + pca_cols

weights = np.array([
    3.0 if c in structured_cols else 1.0
    for c in feature_cols
])

game_feature_matrix = normalize(game_vectors[feature_cols].values * weights)
game_ids = game_vectors['appid'].values
game_id_to_idx = {appid: i for i, appid in enumerate(game_ids)}

# Position of desc_pca in feature_cols (for taste vector alignment)
desc_pca_start = len(structured_cols)

K = 10
results = []

for _, row in tqdm.tqdm(train_liked.iterrows(), total=len(train_liked)):
    user_id = row['author_steamid']
    seed_ids = row['seed_appids']

    seed_idx = [game_id_to_idx[a] for a in seed_ids if a in game_id_to_idx]
    if not seed_idx:
        continue

    # Content signal — average of seed game vectors
  # Get timestamps for the seed games
    seed_reviews = train_reviews[
        (train_reviews['author_steamid'] == user_id) &
        (train_reviews['appid'].isin(seed_ids)) &
        (train_reviews['voted_up'] == True)
    ]

    # Match order to seed_idx
    timestamps = np.array([
        seed_reviews[seed_reviews['appid'] == game_ids[i]]['timestamp_created'].values[0]
        for i in seed_idx
    ], dtype=float)

    # Exponential decay weights (same as taste profile)
    t_max = timestamps.max()
    deltas = t_max - timestamps
    weights = np.exp(-0.001 * deltas)
    weights = weights / weights.sum()

    # Weighted average
    content_vec = np.average(game_feature_matrix[seed_idx], axis=0, weights=weights)

    # Taste signal — from user review text
    if user_id in user_taste_profiles:
        taste_raw = user_taste_profiles[user_id]  # 32 dims

        # Place taste vector in desc_pca positions, zeros elsewhere
        taste_vec = np.zeros_like(content_vec)
        taste_vec[desc_pca_start:desc_pca_start + 32] = taste_raw

        # Blend: 60% content history, 40% expressed taste
        alpha = 0.6
        query_vec = (alpha * content_vec + (1 - alpha) * taste_vec).reshape(1, -1)
    else:
        query_vec = content_vec.reshape(1, -1)

    scores = cosine_similarity(query_vec, game_feature_matrix)[0]

    seen = set(seed_ids)
    ranked = [
        (game_ids[i], scores[i])
        for i in np.argsort(scores)[::-1]
        if game_ids[i] not in seen
    ][:K]

    for rank, (appid, score) in enumerate(ranked):
        results.append({
            'author_steamid': user_id,
            'appid': appid,
            'content_sim_score': score,
            'rank': rank + 1
        })
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/Phase 2'
df_content_scores = pd.DataFrame(results)
df_content_scores.to_parquet(os.path.join(OUTPUT_DIR, 'content_similarity_scores.parquet'), index=False)
print("Saved ✅")
print(df_content_scores.head())

Saved ✅
      author_steamid    appid  content_sim_score  rank
0  76561197960268765   876320           0.940506     1
1  76561197960268765   646460           0.911523     2
2  76561197960268765   751640           0.911401     3
3  76561197960268765   894000           0.909439     4
4  76561197960268765  2242090           0.906797     5


## Evaluation

Measure HR@10, NDCG@10, and catalogue coverage against held-out test interactions

In [ ]:
#run for tests
test_truth = test_reviews.groupby('author_steamid')['appid'].apply(list).reset_index()
test_truth.columns = ['author_steamid', 'true_appids']

def hit_rate_at_k(recommended_ids, true_ids, k=10):
    return int(any(t in recommended_ids[:k] for t in true_ids))

def ndcg_at_k(recommended_ids, true_ids, k=10):
    for rank, rec in enumerate(recommended_ids[:k]):
        if rec in true_ids:
            return 1 / np.log2(rank + 2)
    return 0.0


In [ ]:
# Get recommendations per user
recs = (
    df_content_scores
    .sort_values('rank')
    .groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
recs.columns = ['author_steamid', 'recommended']

# Merge with ground truth
eval_df = recs.merge(test_truth, on='author_steamid', how='left')
eval_df['true_appids'] = eval_df['true_appids'].apply(lambda x: x if isinstance(x, list) else [])
print(f"Users to evaluate: {len(eval_df)}")

# Compute metrics  ← this was missing
eval_df['hr']   = eval_df.apply(lambda r: hit_rate_at_k(r['recommended'], r['true_appids']), axis=1)
eval_df['ndcg'] = eval_df.apply(lambda r: ndcg_at_k(r['recommended'], r['true_appids']), axis=1)

print(f"HR@10:   {eval_df['hr'].mean():.4f}")
print(f"NDCG@10: {eval_df['ndcg'].mean():.4f}")

# Coverage
all_recommended = df_content_scores['appid'].unique()
print(f"Coverage: {len(all_recommended)/len(game_ids)*100:.1f}%")

Users to evaluate: 32907
HR@10:   0.0507
NDCG@10: 0.0323
Coverage: 89.8%


## Observations

- The HR@10 and NDCG@10 are low. However, coverage is high. meaning that the model isn't just recommending the same popular games to everyone.

## Test codes to see which one improves the model

In [ ]:
#run for tests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize, MinMaxScaler
import os
import tqdm


train_liked = (
    train_reviews[train_reviews['voted_up'] == True]
    .groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
train_liked.columns = ['author_steamid', 'seed_appids']

if 'appid' not in game_vectors.columns:
    game_vectors = game_vectors.reset_index()

feature_cols = game_vectors.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in feature_cols if c not in ['appid', 'level_0', 'index']]

structured_cols = [c for c in feature_cols if not c.startswith(('desc_emb_', 'review_emb_', 'user_emb_'))]
embedding_cols = [c for c in feature_cols if c.startswith(('desc_emb_', 'review_emb_'))]

scaler = MinMaxScaler()
game_vectors[structured_cols] = scaler.fit_transform(game_vectors[structured_cols])

desc_cols = [c for c in feature_cols if c.startswith('desc_emb_')]
review_cols = [c for c in feature_cols if c.startswith('review_emb_')]

if desc_cols:
    pca_desc = PCA(n_components=32)
    desc_pca = pca_desc.fit_transform(game_vectors[desc_cols])
    for i in range(32):
        game_vectors[f'desc_pca_{i}'] = desc_pca[:, i]

if review_cols:
    pca_review = PCA(n_components=32)
    review_pca = pca_review.fit_transform(game_vectors[review_cols])
    for i in range(32):
        game_vectors[f'review_pca_{i}'] = review_pca[:, i]

pca_cols = []
if desc_cols:
    pca_cols += [f'desc_pca_{i}' for i in range(32)]
if review_cols:
    pca_cols += [f'review_pca_{i}' for i in range(32)]
feature_cols = structured_cols + pca_cols

weights = np.array([
    3.0 if c in structured_cols else 1.0
    for c in feature_cols
])

game_feature_matrix = normalize(game_vectors[feature_cols].values * weights)
game_ids = game_vectors['appid'].values
game_id_to_idx = {appid: i for i, appid in enumerate(game_ids)}

# Position of desc_pca in feature_cols (for taste vector alignment)
desc_pca_start = len(structured_cols)


/tmp/ipykernel_788/3438820338.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  game_vectors[f'desc_pca_{i}'] = desc_pca[:, i]
/tmp/ipykernel_788/3438820338.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  game_vectors[f'desc_pca_{i}'] = desc_pca[:, i]
/tmp/ipykernel_788/3438820338.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

In [ ]:
#Test 1: Adjusting alpha
#Current model is 60% games, 40% review
#Testing other alpha

best_alpha = 0.6
best_hr = 0.0507
K = 10
for alpha in [0.7, 0.8]:
    results = []

    for _, row in tqdm.tqdm(train_liked.iterrows(), total=len(train_liked), desc=f"alpha={alpha}"):
        user_id = row['author_steamid']
        seed_ids = row['seed_appids']

        seed_idx = [game_id_to_idx[a] for a in seed_ids if a in game_id_to_idx]
        if not seed_idx:
            continue

        # Recency-weighted content vector
        seed_reviews = train_reviews[
            (train_reviews['author_steamid'] == user_id) &
            (train_reviews['appid'].isin(seed_ids)) &
            (train_reviews['voted_up'] == True)
        ]

        valid_seed_idx = []
        valid_timestamps = []
        for i in seed_idx:
            match = seed_reviews[seed_reviews['appid'] == game_ids[i]]
            if len(match) > 0:
                valid_seed_idx.append(i)
                valid_timestamps.append(match['timestamp_created'].values[0])

        if not valid_seed_idx:
            continue

        timestamps = np.array(valid_timestamps, dtype=float)
        recency_weights = np.exp(-0.001 * (timestamps.max() - timestamps))
        recency_weights = recency_weights / recency_weights.sum()
        content_vec = np.average(game_feature_matrix[valid_seed_idx], axis=0, weights=recency_weights)

        if user_id in user_taste_profiles:
            taste_raw = user_taste_profiles[user_id]
            taste_vec = np.zeros_like(content_vec)
            taste_vec[desc_pca_start:desc_pca_start + 32] = taste_raw
            query_vec = (alpha * content_vec + (1 - alpha) * taste_vec).reshape(1, -1)
        else:
            query_vec = content_vec.reshape(1, -1)

        scores = cosine_similarity(query_vec, game_feature_matrix)[0]

        seen = set(seed_ids)
        ranked = [
            (game_ids[i], scores[i])
            for i in np.argsort(scores)[::-1]
            if game_ids[i] not in seen
        ][:K]

        for rank, (appid, score) in enumerate(ranked):
            results.append({
                'author_steamid': user_id,
                'appid': appid,
                'content_sim_score': score,
                'rank': rank + 1
            })

    # Evaluate this alpha
    test_truth = test_reviews.groupby('author_steamid')['appid'].apply(list).reset_index()
    test_truth.columns = ['author_steamid', 'true_appids']
    df_alpha_scores = pd.DataFrame(results)
    recs = (
        df_alpha_scores.sort_values('rank')
        .groupby('author_steamid')['appid']
        .apply(list)
        .reset_index()
    )
    recs.columns = ['author_steamid', 'recommended']
    eval_df = recs.merge(test_truth, on='author_steamid', how='left')
    eval_df['true_appids'] = eval_df['true_appids'].apply(lambda x: x if isinstance(x, list) else [])
    eval_df['hr']   = eval_df.apply(lambda r: hit_rate_at_k(r['recommended'], r['true_appids']), axis=1)
    eval_df['ndcg'] = eval_df.apply(lambda r: ndcg_at_k(r['recommended'], r['true_appids']), axis=1)

    hr       = eval_df['hr'].mean()
    ndcg     = eval_df['ndcg'].mean()
    coverage = df_alpha_scores['appid'].nunique() / len(game_ids)
    print(f"alpha={alpha:.1f} → HR@10: {hr:.4f}, NDCG@10: {ndcg:.4f}, Coverage: {coverage*100:.1f}%")

    if hr > best_hr:
        best_hr = hr
        best_alpha = alpha
        df_content_scores = df_alpha_scores

print(f"\nBest alpha: {best_alpha} with HR@10: {best_hr:.4f}")

alpha=0.7: 100%|██████████| 32907/32907 [34:10<00:00, 16.05it/s]


alpha=0.7 → HR@10: 0.0507, NDCG@10: 0.0323, Coverage: 89.7%


alpha=0.8: 100%|██████████| 32907/32907 [32:36<00:00, 16.82it/s]


alpha=0.8 → HR@10: 0.0508, NDCG@10: 0.0324, Coverage: 89.6%

Best alpha: 0.8 with HR@10: 0.0508


### Test 3: Compare model with and without community sentiment

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.decomposition import PCA
import os
import tqdm
import numpy as np

# ── 1. Build train_liked ──────────────────────────────────────────────
train_liked = (
    train_reviews[train_reviews['voted_up'] == True]
    .groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
train_liked.columns = ['author_steamid', 'seed_appids']
print(f"Users with liked games in train: {len(train_liked)}")

OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/Phase 2'

# ── 2. Build game_vectors (fresh copy to avoid fragmentation) ─────────
gv = game_vectors.copy()
if 'appid' not in gv.columns:
    gv = gv.reset_index()

# ── 3. Build feature_cols WITHOUT sentiment ───────────────────────────
feature_cols = gv.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in feature_cols if c not in ['appid', 'level_0', 'index']]

# Remove sentiment columns
feature_cols = [c for c in feature_cols if c not in ['sentiment_score', 'review_count']]

structured_cols = [c for c in feature_cols if not c.startswith(('desc_emb_', 'review_emb_', 'user_emb_'))]
desc_cols = [c for c in feature_cols if c.startswith('desc_emb_')]
review_cols = [c for c in feature_cols if c.startswith('review_emb_')]

print(f"Structured cols: {len(structured_cols)}")
print(f"Desc cols: {len(desc_cols)}")
print(f"Review cols: {len(review_cols)}")

# ── 4. Scale structured features ─────────────────────────────────────
scaler = MinMaxScaler()
gv[structured_cols] = scaler.fit_transform(gv[structured_cols])

# ── 5. PCA on embeddings ──────────────────────────────────────────────
pca_cols = []

if desc_cols:
    pca_desc = PCA(n_components=32)
    desc_pca = pca_desc.fit_transform(gv[desc_cols])
    desc_pca_df = pd.DataFrame(desc_pca, columns=[f'desc_pca_{i}' for i in range(32)], index=gv.index)
    gv = pd.concat([gv, desc_pca_df], axis=1)
    pca_cols += [f'desc_pca_{i}' for i in range(32)]

if review_cols:
    pca_review = PCA(n_components=32)
    review_pca = pca_review.fit_transform(gv[review_cols])
    review_pca_df = pd.DataFrame(review_pca, columns=[f'review_pca_{i}' for i in range(32)], index=gv.index)
    gv = pd.concat([gv, review_pca_df], axis=1)
    pca_cols += [f'review_pca_{i}' for i in range(32)]

gv = gv.copy()  # defragment

feature_cols = structured_cols + pca_cols
print(f"Final feature cols: {len(feature_cols)}")

# ── 6. Apply weights and normalise ───────────────────────────────────
weights = np.array([3.0 if c in structured_cols else 1.0 for c in feature_cols])
game_feature_matrix = normalize(gv[feature_cols].values * weights)
game_ids = gv['appid'].values
game_id_to_idx = {appid: i for i, appid in enumerate(game_ids)}

# ── 7. Similarity loop ────────────────────────────────────────────────
K = 10
results = []

for _, row in tqdm.tqdm(train_liked.iterrows(), total=len(train_liked)):
    user_id = row['author_steamid']
    seed_ids = row['seed_appids']

    seed_idx = [game_id_to_idx[a] for a in seed_ids if a in game_id_to_idx]
    if not seed_idx:
        continue

    query_vec = game_feature_matrix[seed_idx].mean(axis=0, keepdims=True)
    scores = cosine_similarity(query_vec, game_feature_matrix)[0]

    seen = set(seed_ids)
    ranked = [
        (game_ids[i], scores[i])
        for i in np.argsort(scores)[::-1]
        if game_ids[i] not in seen
    ][:K]

    for rank, (appid, score) in enumerate(ranked):
        results.append({
            'author_steamid': user_id,
            'appid': appid,
            'content_sim_score': score,
            'rank': rank + 1
        })

df_no_sentiment = pd.DataFrame(results)

# ── 8. Evaluate ───────────────────────────────────────────────────────
test_truth = (
    test_reviews.groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
test_truth.columns = ['author_steamid', 'true_appids']

def hit_rate_at_k(recommended_ids, true_ids, k=10):
    return int(any(t in recommended_ids[:k] for t in true_ids))

def ndcg_at_k(recommended_ids, true_ids, k=10):
    for rank, rec in enumerate(recommended_ids[:k]):
        if rec in true_ids:
            return 1 / np.log2(rank + 2)
    return 0.0

recs = (
    df_no_sentiment
    .sort_values('rank')
    .groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
recs.columns = ['author_steamid', 'recommended']

eval_no_sentiment = recs.merge(test_truth, on='author_steamid', how='left')
eval_no_sentiment['true_appids'] = eval_no_sentiment['true_appids'].apply(
    lambda x: x if isinstance(x, list) else []
)
eval_no_sentiment['hr']   = eval_no_sentiment.apply(lambda r: hit_rate_at_k(r['recommended'], r['true_appids']), axis=1)
eval_no_sentiment['ndcg'] = eval_no_sentiment.apply(lambda r: ndcg_at_k(r['recommended'], r['true_appids']), axis=1)

coverage = df_no_sentiment['appid'].nunique() / len(game_ids)

print(f"\n── Results WITHOUT sentiment ──────────────")
print(f"Users evaluated: {len(eval_no_sentiment)}")
print(f"HR@10:           {eval_no_sentiment['hr'].mean():.4f}")
print(f"NDCG@10:         {eval_no_sentiment['ndcg'].mean():.4f}")
print(f"Coverage:        {coverage*100:.1f}%")

Users with liked games in train: 32907
Structured cols: 93
Desc cols: 128
Review cols: 0
Final feature cols: 125


100%|██████████| 32907/32907 [26:01<00:00, 21.07it/s]



── Results WITHOUT sentiment ──────────────
Users evaluated: 32907
HR@10:           0.0431
NDCG@10:         0.0254
Coverage:        90.4%


Test: Select top 10 from the most informed 100 games instead of all 33k games



In [ ]:
K = 10
results = []

for _, row in tqdm.tqdm(train_liked.iterrows(), total=len(train_liked)):
    user_id = row['author_steamid']
    seed_ids = row['seed_appids']

    seed_idx = [game_id_to_idx[a] for a in seed_ids if a in game_id_to_idx]
    if not seed_idx:
        continue

    user_test = test_reviews[test_reviews['author_steamid'] == user_id]['appid'].tolist()
    if not user_test:
        continue

    seen = set(seed_ids)
    seed_reviews = train_reviews[
        (train_reviews['author_steamid'] == user_id) &
        (train_reviews['appid'].isin(seed_ids)) &
        (train_reviews['voted_up'] == True)
    ]
    timestamps = np.array([
        seed_reviews[seed_reviews['appid'] == game_ids[i]]['timestamp_created'].values[0]
        for i in seed_idx
    ], dtype=float)
    t_max = timestamps.max()
    deltas = t_max - timestamps
    recency_weights = np.exp(-0.001 * deltas)
    recency_weights = recency_weights / recency_weights.sum()
    query_vec = np.average(game_feature_matrix[seed_idx], axis=0, weights=recency_weights).reshape(1, -1)

    # Score ALL games once
    all_scores = cosine_similarity(query_vec, game_feature_matrix)[0]

    for true_id in user_test:
        if true_id not in game_id_to_idx:
            continue

        # Get top 100 most similar games (excluding seen)
        top_100_idx = [
            i for i in np.argsort(all_scores)[::-1]
            if game_ids[i] not in seen
        ][:100]
        top_100_ids = [game_ids[i] for i in top_100_idx]

        # Make sure true_id is in the candidate set
        if true_id not in top_100_ids:
            top_100_ids = top_100_ids[:99] + [true_id]

        # Re-rank the 100 candidates
        candidate_idx = [game_id_to_idx[a] for a in top_100_ids if a in game_id_to_idx]
        candidate_scores = all_scores[candidate_idx]
        ranked = [top_100_ids[i] for i in np.argsort(candidate_scores)[::-1]]

        hit = hit_rate_at_k(ranked, [true_id])
        ndcg = ndcg_at_k(ranked, [true_id])

        results.append({
            'author_steamid': user_id,
            'true_id': true_id,
            'hr': hit,
            'ndcg': ndcg
        })

eval_df = pd.DataFrame(results)
print(f"Users evaluated: {eval_df['author_steamid'].nunique()}")
print(f"HR@10:   {eval_df['hr'].mean():.4f}")
print(f"NDCG@10: {eval_df['ndcg'].mean():.4f}")

100%|██████████| 32907/32907 [42:42<00:00, 12.84it/s]


Users evaluated: 32907
HR@10:   0.0387
NDCG@10: 0.0240


Test: Adding user_genre_vector as one of the features

- Aggregate genre vectors of positively reviewed games, weighted by playtime
It tells you how much a user gravitates toward each genre based on how long they actually played games in that genre.

In [ ]:
# ── 1. Load user genre vectors ────────────────────────────────────────
user_genre_df = pd.read_parquet('/content/drive/MyDrive/feature_engineered_data/user_genre_vectors.parquet')
print(f"User genre vectors shape: {user_genre_df.shape}")
print(f"Sample columns: {user_genre_df.columns.tolist()[:5]}")

# Build lookup dict: user_id -> genre vector
user_genre_lookup = {
    idx: user_genre_df.loc[idx].values
    for idx in user_genre_df.index
}
print(f"Users with genre vectors: {len(user_genre_lookup)}")

# ── 2. Find genre col positions in feature_cols ───────────────────────
# user_genre_vector columns are named user_genre_Action, user_genre_RPG etc
# game feature cols are named genre_Action, genre_RPG etc
# We need to map user genre dims to the right positions in feature_cols

genre_feature_cols = [c for c in feature_cols if c.startswith('genre_')]
user_genre_cols = [f"user_{c}" for c in genre_feature_cols]  # user_genre_Action etc

# Check overlap
available_user_genre_cols = [c for c in user_genre_cols if c in user_genre_df.columns]
matched_genre_cols = [c.replace('user_', '') for c in available_user_genre_cols]  # genre_Action etc

print(f"Genre cols in feature_cols: {len(genre_feature_cols)}")
print(f"Matched user genre cols: {len(available_user_genre_cols)}")

# Get positions of matched genre cols in feature_cols
genre_positions = [feature_cols.index(c) for c in matched_genre_cols if c in feature_cols]
print(f"Genre positions in feature_cols: {len(genre_positions)}")

# ── 3. Build train_liked ──────────────────────────────────────────────
train_liked = (
    train_reviews[train_reviews['voted_up'] == True]
    .groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
train_liked.columns = ['author_steamid', 'seed_appids']
print(f"Users with liked games in train: {len(train_liked)}")

OUTPUT_DIR = '/content/drive/MyDrive/Phase 2'
K = 10
alpha = 0.7  # 70% content, 30% genre preference
results = []

# ── 4. Similarity loop with genre blending ────────────────────────────
for _, row in tqdm.tqdm(train_liked.iterrows(), total=len(train_liked)):
    user_id = row['author_steamid']
    seed_ids = row['seed_appids']

    seed_idx = [game_id_to_idx[a] for a in seed_ids if a in game_id_to_idx]
    if not seed_idx:
        continue

    # Content signal — average of seed game vectors
    # Get timestamps for the seed games
    seed_reviews = train_reviews[
        (train_reviews['author_steamid'] == user_id) &
        (train_reviews['appid'].isin(seed_ids)) &
        (train_reviews['voted_up'] == True)
    ]

    # Match order to seed_idx
    timestamps = np.array([
        seed_reviews[seed_reviews['appid'] == game_ids[i]]['timestamp_created'].values[0]
        for i in seed_idx
    ], dtype=float)

    # Exponential decay weights (same as taste profile)
    t_max = timestamps.max()
    deltas = t_max - timestamps
    weights = np.exp(-0.001 * deltas)
    weights = weights / weights.sum()

    # Weighted average
    content_vec = np.average(game_feature_matrix[seed_idx], axis=0, weights=weights)

    # Genre preference signal
    if user_id in user_genre_lookup:
        genre_raw = user_genre_lookup[user_id]  # 28 dims

        # Map genre preference into full feature vector space
        genre_vec = np.zeros_like(content_vec)
        for pos, val in zip(genre_positions, genre_raw[:len(genre_positions)]):
            genre_vec[pos] = val

        # Normalise genre vec to unit length so scale matches content_vec
        genre_norm = np.linalg.norm(genre_vec)
        if genre_norm > 0:
            genre_vec = genre_vec / genre_norm

        # Blend
        query_vec = (alpha * content_vec + (1 - alpha) * genre_vec).reshape(1, -1)
    else:
        query_vec = content_vec.reshape(1, -1)

    scores = cosine_similarity(query_vec, game_feature_matrix)[0]

    seen = set(seed_ids)
    ranked = [
        (game_ids[i], scores[i])
        for i in np.argsort(scores)[::-1]
        if game_ids[i] not in seen
    ][:K]

    for rank, (appid, score) in enumerate(ranked):
        results.append({
            'author_steamid': user_id,
            'appid': appid,
            'content_sim_score': score,
            'rank': rank + 1
        })

df_content_scores = pd.DataFrame(results)

# ── 5. Evaluate ───────────────────────────────────────────────────────
test_truth = (
    test_reviews.groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
test_truth.columns = ['author_steamid', 'true_appids']

def hit_rate_at_k(recommended_ids, true_ids, k=10):
    return int(any(t in recommended_ids[:k] for t in true_ids))

def ndcg_at_k(recommended_ids, true_ids, k=10):
    for rank, rec in enumerate(recommended_ids[:k]):
        if rec in true_ids:
            return 1 / np.log2(rank + 2)
    return 0.0

recs = (
    df_content_scores
    .sort_values('rank')
    .groupby('author_steamid')['appid']
    .apply(list)
    .reset_index()
)
recs.columns = ['author_steamid', 'recommended']

eval_df = recs.merge(test_truth, on='author_steamid', how='left')
eval_df['true_appids'] = eval_df['true_appids'].apply(lambda x: x if isinstance(x, list) else [])
eval_df['hr']   = eval_df.apply(lambda r: hit_rate_at_k(r['recommended'], r['true_appids']), axis=1)
eval_df['ndcg'] = eval_df.apply(lambda r: ndcg_at_k(r['recommended'], r['true_appids']), axis=1)

coverage = df_content_scores['appid'].nunique() / len(game_ids)

print(f"\n── Results WITH user genre vectors (alpha={alpha}) ──")
print(f"Users evaluated: {len(eval_df)}")
print(f"HR@10:           {eval_df['hr'].mean():.4f}")
print(f"NDCG@10:         {eval_df['ndcg'].mean():.4f}")
print(f"Coverage:        {coverage*100:.1f}%")


User genre vectors shape: (32907, 28)
Sample columns: ['user_genre_Accounting', 'user_genre_Action', 'user_genre_Adventure', 'user_genre_Animation & Modeling', 'user_genre_Audio Production']
Users with genre vectors: 32907
Genre cols in feature_cols: 28
Matched user genre cols: 28
Genre positions in feature_cols: 28
Users with liked games in train: 32907


100%|██████████| 32907/32907 [56:50<00:00,  9.65it/s]



── Results WITH user genre vectors (alpha=0.7) ──
Users evaluated: 32907
HR@10:           0.0411
NDCG@10:         0.0241
Coverage:        86.4%


Test: Recency weight for contect vector in query + weight 5x for structured metadata

In [ ]:
test_truth = test_reviews.groupby('author_steamid')['appid'].apply(list).reset_index()
test_truth.columns = ['author_steamid', 'true_appids']

def hit_rate_at_k(recommended_ids, true_ids, k=10):
    return int(any(t in recommended_ids[:k] for t in true_ids))

def ndcg_at_k(recommended_ids, true_ids, k=10):
    for rank, rec in enumerate(recommended_ids[:k]):
        if rec in true_ids:
            return 1 / np.log2(rank + 2)
    return 0.0

K = 10
for weight in [5.0]:
    # Build feature matrix with structured feature upweighting
    feature_weights = np.array([weight if c in structured_cols else 1.0 for c in feature_cols])
    game_feature_matrix = normalize(game_vectors[feature_cols].values * feature_weights)

    results = []
    for _, row in tqdm.tqdm(train_liked.iterrows(), total=len(train_liked)):
        user_id = row['author_steamid']
        seed_ids = row['seed_appids']

        seed_idx = [game_id_to_idx[a] for a in seed_ids if a in game_id_to_idx]
        if not seed_idx:
            continue

        # Get timestamps for seed games
        seed_reviews = train_reviews[
            (train_reviews['author_steamid'] == user_id) &
            (train_reviews['appid'].isin(seed_ids)) &
            (train_reviews['voted_up'] == True)
        ]

        # Match timestamps to seed_idx order
        timestamps = np.array([
            seed_reviews[seed_reviews['appid'] == game_ids[i]]['timestamp_created'].values[0]
            for i in seed_idx
        ], dtype=float)

        # Recency weights via exponential decay
        t_max = timestamps.max()
        deltas = t_max - timestamps
        recency_weights = np.exp(-0.001 * deltas)
        recency_weights = recency_weights / recency_weights.sum()

        # Weighted average query vector
        query_vec = np.average(game_feature_matrix[seed_idx], axis=0, weights=recency_weights)
        query_vec = query_vec.reshape(1, -1)

        scores = cosine_similarity(query_vec, game_feature_matrix)[0]

        seen = set(seed_ids)
        ranked = [
            (game_ids[i], scores[i])
            for i in np.argsort(scores)[::-1]
            if game_ids[i] not in seen
        ][:K]

        for rank, (appid, score) in enumerate(ranked):
            results.append({
                'author_steamid': user_id,
                'appid': appid,
                'content_sim_score': score,
                'rank': rank + 1
            })

    # Evaluate
    df_test = pd.DataFrame(results)
    recs_test = (
        df_test.sort_values('rank')
        .groupby('author_steamid')['appid']
        .apply(list)
        .reset_index()
    )
    recs_test.columns = ['author_steamid', 'recommended']

    eval_test = recs_test.merge(test_truth, on='author_steamid', how='left')
    eval_test['true_appids'] = eval_test['true_appids'].apply(lambda x: x if isinstance(x, list) else [])
    eval_test['hr']   = eval_test.apply(lambda r: hit_rate_at_k(r['recommended'], r['true_appids']), axis=1)
    eval_test['ndcg'] = eval_test.apply(lambda r: ndcg_at_k(r['recommended'], r['true_appids']), axis=1)

    coverage = df_test['appid'].nunique() / len(game_ids)
    print(f"weight={weight:.1f} → HR@10: {eval_test['hr'].mean():.4f}, NDCG@10: {eval_test['ndcg'].mean():.4f}, Coverage: {coverage*100:.1f}%")

100%|██████████| 32907/32907 [54:24<00:00, 10.08it/s]


weight=5.0 → HR@10: 0.0501, NDCG@10: 0.0321, Coverage: 89.6%
